<a href="https://colab.research.google.com/github/Kmjndv/VeritasAIHackathon/blob/main/Respira.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Respira
### Our approach to respiratory sound classification

We turn uploaded digital-stethoscope chest recordings into five respiratory sound categories, then present accepted findings with optional symptom context in our browser app.

**[Our live app · Respira](https://respira-breathe-well.smart-moss-9631.chatgpt.site)**

We use uploaded recordings in our current demo, and we run audio inference in the browser. We can demonstrate the app without running this notebook.

| Our input | Our model | Our output |
|:---|:---|:---|
| Uploaded stethoscope audio | Four-second log-mel features and our selected CNN | Category scores, accepted findings, and contextual explanations |

**Our notebook guide**  
[01 · Data](#scrollTo=respira-0fd4b13ef5cb) · [02 · Pipeline](#scrollTo=respira-0f594cc07225) · [03 · Training](#scrollTo=respira-04a94c2de771) · [04 · Plots](#scrollTo=respira-e03f861c0c72) · [05 · Export](#scrollTo=respira-018f400c83fd) · [06 · Evidence](#scrollTo=respira-1fd1f1fae57b) · [07 · Next steps](#scrollTo=respira-8bf0a8185052)

---


## 00 · Our approach and setup

We classify **Asthma, Healthy, Bronchial, COPD, and Pneumonia** with our selected `wide-crops` CNN, version `respira-cnn-v4-dataset-only`. We use the dataset's category names throughout our training and evaluation.

| Our design choice | Our implementation |
|:---|:---|
| Audio representation | Mono audio at 8 kHz; four-second windows |
| Spectrogram | 32 log-mel bands spanning 60–1,800 Hz |
| Model selection | Validation macro F1 across three CNN candidates and two ensembles |
| Score calibration | Temperature scaling on validation recordings |
| Category acceptance | Audio-quality checks, training-data similarity, score margin, and class thresholds |
| Deployment | ONNX weights with matching preprocessing and acceptance settings |

We train on public respiratory recordings only. We do not train a background-noise category or use external noise recordings in our current CNN.

We run the main workflow from top to bottom. We create the working folders and install the required packages below. We keep `RUN_LEGACY_NOTEBOOK = False` so our archived experiments remain separate from our current results.


In [ ]:
# We prepare our environment and keep our archived experiments disabled.
RUN_LEGACY_NOTEBOOK = False
%pip install numpy scipy scikit-learn soundfile torch onnx onnxruntime pandas matplotlib
from pathlib import Path
import os, json, sys, subprocess, zipfile, urllib.request
ROOT=Path('/content/respira')
ROOT.mkdir(exist_ok=True)
(ROOT/'ml').mkdir(exist_ok=True)
(ROOT/'data').mkdir(exist_ok=True)
os.chdir(ROOT)


## 01 · Our dataset

We use Mohammed Tawfik Musaed's [Asthma Detection Dataset](https://www.kaggle.com/datasets/mohammedtawfikmusaed/asthma-detection-dataset-version-2). We begin with **1,211 recordings** and retain **1,199 unique recordings** after removing exact duplicate audio during training.

| Our split | Unique recordings | Our use |
|:---|---:|:---|
| Training | 715 | We fit model weights and training-data descriptors. |
| Validation | 265 | We choose checkpoints, compare candidates, calibrate scores, and set thresholds. |
| Internal test | 219 | We evaluate our selected model and fixed acceptance rules. |

We keep matching `P`-number filename groups in the same split to reduce overlap. We have not independently verified patient identities, and we inspected this internal test set during earlier development. We therefore report an internal benchmark, not independent external validation.

We reuse an available dataset folder or download the archive if needed. We count the extracted WAV files below before deduplication.


In [ ]:
# We locate our respiratory dataset and verify all five category folders.
required = {'bronchial', 'asthma', 'copd', 'healthy', 'pneumonia'}
candidates = [Path('/content/Asthma_Detection_Dataset'), ROOT/'data'/'asthma-v2']
# We reuse the dataset path if it is already set.
if 'dataset_path' in globals():
    candidates.insert(0, Path(dataset_path))
RESPIRATORY_ROOT = None
for candidate in candidates:
    if candidate.exists():
        folder_labels = {p.parent.name.lower() for p in candidate.rglob('*.wav')}
        if required <= folder_labels:
            RESPIRATORY_ROOT = candidate.resolve()
            break
if RESPIRATORY_ROOT is None:
    archive = ROOT/'data'/'asthma-v2.zip'
    if not archive.exists():
        urllib.request.urlretrieve('https://www.kaggle.com/api/v1/datasets/download/mohammedtawfikmusaed/asthma-detection-dataset-version-2', archive)
    destination = (ROOT/'data'/'asthma-v2').resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            target = (destination/item.filename).resolve()
            if destination not in target.parents:
                raise ValueError('Unsafe archive path')
            z.extract(item, destination)
    RESPIRATORY_ROOT = destination
print('Using existing/extracted respiratory audio:', RESPIRATORY_ROOT)
from collections import Counter
counts = Counter(p.parent.name for p in RESPIRATORY_ROOT.rglob('*.wav'))
print('All extracted respiratory files:', sum(counts.values()))
display(__import__('pandas').DataFrame(sorted(counts.items()), columns=['Category', 'Recordings']))


Using existing/extracted respiratory audio: /content/respira/data/asthma-v2
All extracted respiratory files: 1211


,Category,Recordings
0,Bronchial,104
1,asthma,288
2,copd,401
3,healthy,133
4,pneumonia,285


## 02 · Our model pipeline
### 02.1 · We prepare the audio

We convert each recording to mono, resample it to **8 kHz**, and extract a **four-second window**. We pad short clips when constructing model inputs; for our first-window test audit, we require a complete four seconds.

We remove the waveform's mean, normalize its peak amplitude, and calculate a **32-band log-mel spectrogram** with a 512-sample FFT and 256-sample hop. We then standardize the spectrogram before classification.

We save these shared preprocessing functions in `ml/audio_features.py` so our training and audit use the same representation.


In [ ]:
%%writefile /content/respira/ml/audio_features.py
"""We share four-second audio preprocessing across our training and audit."""
import math
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
SR=8000;SECONDS=4;N=SR*SECONDS;FFT=512;HOP=256;MELS=32;FRAMES=1+(N-FFT)//HOP
SEED=42

def melbank():
 """We build 32 triangular mel filters covering 60–1,800 Hz."""
 hz=lambda m:700*(10**(m/2595)-1)
 points=hz(np.linspace(2595*np.log10(1+60/700),2595*np.log10(1+1800/700),MELS+2))
 bins=np.arange(FFT//2+1)*SR/FFT
 return np.array([np.maximum(0,np.minimum((bins-a)/(b-a),(c-bins)/(c-b))) for a,b,c in zip(points,points[1:],points[2:])],dtype=np.float32)
BANK=melbank();WINDOW=np.hanning(FFT+1)[:-1].astype(np.float32)
def read_audio(path):
 """We mix channels to mono and resample to 8 kHz."""
 x,sr=sf.read(path,dtype='float32',always_2d=True);x=x.mean(1)
 g=math.gcd(sr,SR)
 if sr!=SR:x=resample_poly(x,SR//g,sr//g)
 return x.astype(np.float32)
def crop(x,start=0):
 """We take four seconds from our chosen offset and zero-pad short clips."""
 y=x[start:start+N]
 return np.pad(y,(0,max(0,N-len(y)))).astype(np.float32)
def features(x):
 """We convert a four-second waveform to a normalized log-mel spectrogram."""
 x=x-np.mean(x);peak=np.max(np.abs(x))
 if peak>1e-8:x=x/peak
 frames=np.lib.stride_tricks.sliding_window_view(x,FFT)[::HOP]
 power=np.abs(np.fft.rfft(frames*WINDOW,axis=1))**2
 spec=np.log(np.maximum(BANK@power.T,1e-8)).astype(np.float32)
 return ((spec-spec.mean())/max(float(spec.std()),1e-4))[None]

def softmax(z):
 """We convert logits to class scores with a numerically stable softmax."""
 z=z-z.max(1,keepdims=True);p=np.exp(z);return p/p.sum(1,keepdims=True)


Overwriting /content/respira/ml/audio_features.py


### 02.2 · We define training and selection

We compare three CNN configurations and two equal-weight ensembles. We use class-balanced sampling, time and frequency masking, AdamW, label smoothing, and validation checkpoint selection. We add random training crops to the `wide-crops` and `compact` candidates.

We rank candidates by **validation macro F1** to give each category equal weight in selection. We then calibrate the selected model's scores and set acceptance thresholds before evaluating our internal test split.

We target **95% validation precision with at least 10 accepted predictions per class**. We disable acceptance for a class when no threshold meets both requirements. We treat 95% as our selection target, not a promised test result.

We save the training workflow in `ml/train_respiratory.py`; we start the run in section 03.


In [ ]:
%%writefile /content/respira/ml/train_respiratory.py
"""We train and compare CNNs on our respiratory dataset, then audit fixed choices.
We use no ESC-50 audio and train no background-noise category.
We choose models and thresholds on validation data, then evaluate our reused
internal test split. We retain its earlier-development limitation.
"""
import argparse,copy,hashlib,json,math,random,re,time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from scipy.optimize import minimize_scalar
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score,balanced_accuracy_score
from audio_features import read_audio,crop,features,softmax,SR,N,FFT,HOP,MELS,FRAMES,BANK
LABELS=['asthma','healthy','bronchial','copd','pneumonia']
class CNN(nn.Module):
 def __init__(self,width=2,dropout=.25):
  super().__init__();layers=[];channels=1
  for out in [8*width,16*width,32*width]:
   layers.extend([nn.Conv2d(channels,out,3,padding=1),nn.BatchNorm2d(out),nn.ReLU(),nn.MaxPool2d(2)]);channels=out
  self.conv=nn.Sequential(*layers);self.head=nn.Sequential(nn.Linear(256*width,64*width),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64*width,len(LABELS)))
 def forward(self,x):
  z=self.conv(x);return self.head(torch.cat([z.mean(3),z.amax(3)],dim=1).flatten(1))
class Ensemble(nn.Module):
 def __init__(self,models):super().__init__();self.models=nn.ModuleList(models)
 def forward(self,x):return torch.stack([m(x) for m in self.models]).mean(0)
def metrics(y,p):
 pred=p.argmax(1)
 return {'accuracy':float(accuracy_score(y,pred)),'balanced_accuracy':float(balanced_accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,average='macro')),'confusion_matrix':confusion_matrix(y,pred,labels=range(5)).tolist(),'report':classification_report(y,pred,labels=range(5),target_names=LABELS,output_dict=True,zero_division=0)}
def predict(model,x):
 model.eval()
 with torch.no_grad():return np.concatenate([model(torch.from_numpy(chunk)).numpy() for chunk in np.array_split(x,max(1,math.ceil(len(x)/64)))])
def nll(y,p):return float(-np.log(p[np.arange(len(y)),y]+1e-9).mean())
def descriptor(x):
 z=x[:,0];return np.concatenate([z.mean(2),z.std(2)],axis=1)
def distances(x,gate):
 d=descriptor(x);return np.sqrt(np.min(np.mean(((d[:,None,:]-np.array(gate['centroids'])[None,:,:])/np.array(gate['scale']))**2,axis=2),axis=1))
def selective(y,p,accepted):
 pred=p.argmax(1);by={}
 for c,name in enumerate(LABELS):
  selected=accepted&(pred==c);tp=int(((y==c)&selected).sum());count=int(selected.sum())
  by[name]={'precision':tp/count if count else None,'accepted_predictions':count,'true_positives':tp,'recall_after_abstention':tp/int((y==c).sum()) if (y==c).any() else None}
 return {'accuracy':float((pred[accepted]==y[accepted]).mean()) if accepted.any() else None,'coverage':float(accepted.mean()),'accepted':int(accepted.sum()),'total':len(y),'per_class':by}
def main():
 ap=argparse.ArgumentParser();ap.add_argument('--data',type=Path,required=True);ap.add_argument('--out',type=Path,required=True);ap.add_argument('--respiratory-root',type=Path);args=ap.parse_args()
 torch.set_num_threads(4);torch.manual_seed(42);random.seed(42);np.random.seed(42);rng=np.random.default_rng(42)
 out=args.out;out.mkdir(parents=True,exist_ok=True);rows=[];waves=[];seen={};duplicates=0
 root=args.respiratory_root or args.data/'asthma-v2'
 # We prepare our data: load WAV files and remove exact duplicate audio.
 for path in sorted(root.rglob('*.wav')):
  name=path.parent.name.lower()
  if name not in LABELS:continue
  y=LABELS.index(name);x=read_audio(path);digest=hashlib.sha256(x.tobytes()).hexdigest()
  if digest in seen:
   if seen[digest]!=y:raise ValueError('Conflicting duplicate labels')
   duplicates+=1;continue
  seen[digest]=y;rows.append({'path':str(path.resolve()),'source':'asthma-v2','label':y,'group':'resp-'+re.match(r'P\d+',path.name).group(),'seconds':len(x)/SR,'sha256_pcm':digest});waves.append(x)
 y=np.array([r['label'] for r in rows]);groups=np.array([r['group'] for r in rows]);split=np.full(len(rows),'train',dtype=object)
 # We define our splits: keep matching filename groups together.
 folds=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(np.arange(len(y)),np.where(y>=2,3,y),groups));split[folds[0][1]]='test';split[folds[1][1]]='validation'
 for a,b in [('train','test'),('train','validation'),('validation','test')]:assert not set(groups[split==a])&set(groups[split==b])
 for i,r in enumerate(rows):r['split']=split[i]
 (out/'manifest.json').write_text(json.dumps(rows,indent=2));print('Dataset only:',len(rows),'unique files; split:',{s:int((split==s).sum()) for s in ['train','validation','test']},flush=True)
 first=np.stack([features(crop(x)) for x in waves]);last=np.stack([features(crop(x,max(0,len(x)-N))) for x in waves]);ids=np.where(split=='train')[0];vi=np.where(split=='validation')[0];ti=np.where(split=='test')[0]
 # We expand our training inputs with random crops from training recordings only.
 crops=np.stack([features(crop(waves[i],int(rng.integers(0,max(1,len(waves[i])-N+1))))) for i in ids for _ in range(2)])
 candidates=[{'name':'wide','width':2,'dropout':.25,'lr':.001,'random_crops':False,'seed':42},{'name':'wide-crops','width':2,'dropout':.35,'lr':.0007,'random_crops':True,'seed':43},{'name':'compact','width':1,'dropout':.25,'lr':.001,'random_crops':True,'seed':44}]
 models=[];records=[];vzs=[]
 # We train our candidates with class balancing and validation checkpoints.
 for spec in candidates:
  torch.manual_seed(spec['seed']);rng=np.random.default_rng(spec['seed']);x=np.concatenate([first[ids],last[ids]]+([crops] if spec['random_crops'] else []));ty=np.concatenate([y[ids],y[ids]]+([np.repeat(y[ids],2)] if spec['random_crops'] else []));weights=1/np.bincount(ty,minlength=5)
  sampler=WeightedRandomSampler(torch.tensor(weights[ty]),len(ty),replacement=True);loader=DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(ty).long()),batch_size=48,sampler=sampler)
  model=CNN(spec['width'],spec['dropout']);optimizer=torch.optim.AdamW(model.parameters(),lr=spec['lr'],weight_decay=.005);scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,80);best=-1;best_epoch=0;history=[];started=time.time()
  for epoch in range(80):
   model.train();total=0
   for xb,yb in loader:
    if rng.random()<.6:
     f=int(rng.integers(MELS-2));xb[:,:,f:f+2,:]=0
    if rng.random()<.6:
     t=int(rng.integers(FRAMES-8));xb[:,:,:,t:t+8]=0
    optimizer.zero_grad();logits=model(xb);loss=nn.functional.cross_entropy(logits,yb,label_smoothing=.03);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5);optimizer.step();total+=loss.item()*len(yb)
   scheduler.step();vz=predict(model,first[vi]);p=softmax(vz);m=metrics(y[vi],p);score=m['macro_f1'];history.append({'epoch':epoch+1,'training_augmented_loss':total/len(ty),'validation_loss':nll(y[vi],p),'validation_accuracy':m['accuracy'],'validation_macro_f1':score})
   print(spec['name'],'epoch',epoch+1,'loss',round(total/len(ty),3),'val_loss',round(history[-1]['validation_loss'],3),'val_acc',round(m['accuracy'],3),'val_f1',round(score,3),flush=True)
   if score>best:best=score;best_epoch=epoch+1;best_state=copy.deepcopy(model.state_dict())
   if epoch+1-best_epoch>=16 and epoch>=35:break
  model.load_state_dict(best_state);model.eval();torch.save(best_state,out/(spec['name']+'.pt'));vz=predict(model,first[vi]);models.append(model);vzs.append(vz)
  records.append({'name':spec['name'],'parameters':spec,'best_epoch':best_epoch,'epochs_run':len(history),'training_seconds':time.time()-started,'history':history,'validation':metrics(y[vi],softmax(vz))})
 # We select our model by comparing validation macro F1 across candidates.
 choices=[([i],vzs[i]) for i in range(len(models))]+[([0,1],(vzs[0]+vzs[1])/2),([0,1,2],np.mean(vzs,axis=0))]
 scored=[{'members':ix,'validation':metrics(y[vi],softmax(z))} for ix,z in choices];win=max(range(len(scored)),key=lambda i:scored[i]['validation']['macro_f1']);chosen=scored[win]['members'];model=Ensemble([models[i] for i in chosen]).eval();vz=choices[win][1]
 selection={'criterion':'Validation first-window macro F1, then fixed equal-weight ensemble comparison','candidates':records,'choices':scored,'selected_members':[candidates[i]['name'] for i in chosen],'test_used_for_selection':False};(out/'model_selection.json').write_text(json.dumps(selection,indent=2));print('SELECTION FROZEN',selection['selected_members'],flush=True)
 # We calibrate our selected model with validation temperature scaling.
 temp=float(minimize_scalar(lambda t:nll(y[vi],softmax(vz/t)),bounds=(.5,5),method='bounded').x);vp=softmax(vz/temp)
 # We estimate similarity to training audio; we do not measure noise detection.
 d=descriptor(first[ids]);scale=np.maximum(d.std(0),.1);centroids=[d[y[ids]==c].mean(0).tolist() for c in range(5)];gate={'centroids':centroids,'scale':scale.tolist()};vd=distances(first[vi],gate);gate['maxDistance']=float(np.quantile(vd,.99));gate['note']='Dataset-support heuristic, not a validated noise detector; fitted on train and validation respiratory recordings only.'
 margin=.15;vsort=np.sort(vp,axis=1);eligible=(vd<=gate['maxDistance'])&(vsort[:,-1]-vsort[:,-2]>=margin);thresholds=[];threshold_details=[]
 # We set our class thresholds using validation precision and sample counts.
 for cls in range(5):
  good=[]
  for threshold in np.arange(.5,1.0,.025):
   accepted=eligible&(vp.argmax(1)==cls)&(vp[:,cls]>=threshold);count=int(accepted.sum());precision=float((y[vi][accepted]==cls).mean()) if count else 0
   if count>=10 and precision>=.95:good.append((float(threshold),count,precision))
  threshold,count,precision=min(good) if good else (1.01,0,None);thresholds.append(threshold);threshold_details.append({'label':LABELS[cls],'threshold':threshold,'validation_accepted':count,'validation_precision':precision,'enabled':bool(good)})
 def accept(x,p):
  sp=np.sort(p,axis=1);return (distances(x,gate)<=gate['maxDistance'])&(sp[:,-1]-sp[:,-2]>=margin)&(p.max(1)>=np.array(thresholds)[p.argmax(1)])
 # We evaluate our internal test split after fixing the model and thresholds.
 zfirst=predict(model,first[ti]);zlast=predict(model,last[ti]);tp=softmax((zfirst+zlast)/2/temp);fp=softmax(zfirst/temp);complete=np.array([len(waves[i])>=N for i in ti]);fids=ti[complete];fp=fp[complete];eligible_first=accept(first[fids],fp);trainp=softmax(predict(model,first[ids])/temp)
 allx=[];ally=[];windowids=[]
 for i in ti:
  for end in range(N,len(waves[i])+1,SR):allx.append(features(waves[i][end-N:end]));ally.append(y[i]);windowids.append(int(i))
 allx=np.stack(allx);ally=np.array(ally);wp=softmax(predict(model,allx)/temp);wa=accept(allx,wp)
 report={'version':'respira-cnn-v4-dataset-only','labels':LABELS,'recordings':len(rows),'external_training_recordings':0,'respiratory_exact_duplicates_removed':duplicates,'split_counts':{s:np.bincount(y[split==s],minlength=5).tolist() for s in ['train','validation','test']},'split_policy':'Unchanged global P-number proxy groups; exact PCM duplicates removed. P-number identity is not independently verified. Existing test set has been inspected in earlier development, not external validation.','selected_members':selection['selected_members'],'model_selection':selection,'validation':metrics(y[vi],vp),'test':metrics(y[ti],tp),'first_window_evaluation':{'seconds':4,'recordings':len(fids),'test':metrics(y[fids],fp),'selective':selective(y[fids],fp,eligible_first)},'window_evaluation':{'window_seconds':4,'hop_seconds':1,'test_windows':len(ally),'test_recordings':len(set(windowids)),'test':metrics(ally,wp),'selective':selective(ally,wp,wa),'note':'Overlapping windows are correlated, not independent participants.'},'selective_policy':{'target_validation_precision':.95,'minimum_validation_predictions_per_class':10,'threshold_details':threshold_details,'margin':margin,'validation':selective(y[vi],vp,accept(first[vi],vp)),'note':'95% is a validation selection target, not a guaranteed precision. Unsupported classes remain disabled. Coverage and per-class recall include the cost of abstention.'},'fit_diagnostics':{'train_first_window_loss':nll(y[ids],trainp),'validation_first_window_loss':nll(y[vi],vp),'train_first_window_accuracy':metrics(y[ids],trainp)['accuracy'],'validation_first_window_accuracy':metrics(y[vi],vp)['accuracy'],'note':'Comparable unaugmented first-window losses after temperature calibration. Curves and train/validation gaps are evidence, not a guarantee of no overfitting or underfitting.'},'temperature':temp,'thresholds':thresholds,'limitations':['No noise-labeled examples: there is no learned background-noise class.','Dataset support rejection cannot reliably distinguish noise from unfamiliar respiratory recordings.','Phone microphone and cough diagnosis are unvalidated; source recordings include stethoscope lung sounds.','Clinical precision is unknown. Internal thresholds may not generalize to real-world audio.','Healthy is a dataset label and cannot confirm health.']}
 report['accuracy_target']={'target':.90,'achieved':bool(report['test']['accuracy']>=.90 and report['first_window_evaluation']['test']['accuracy']>=.90)}
 dummy=torch.from_numpy(first[:1]);torch.onnx.export(model,dummy,str(out/'respira.onnx'),input_names=['features'],output_names=['logits'],opset_version=17,dynamo=False)
 import onnxruntime as ort
 options=ort.SessionOptions();options.intra_op_num_threads=1;session=ort.InferenceSession(str(out/'respira.onnx'),sess_options=options,providers=['CPUExecutionProvider']);actual=session.run(None,{'features':first[:1]})[0];report['onnx_max_abs_error']=float(np.max(np.abs(actual-model(dummy).detach().numpy())))
 times=[]
 for _ in range(30):
  start=time.perf_counter();session.run(None,{'features':first[:1]});times.append((time.perf_counter()-start)*1000)
 report['local_onnx_p50_ms']=float(np.median(times))
 config={'version':report['version'],'labels':LABELS,'sampleRate':SR,'windowSeconds':4,'hopSeconds':1,'fftSize':FFT,'hopSize':HOP,'melBands':MELS,'frames':FRAMES,'melBank':BANK.tolist(),'temperature':temp,'thresholds':thresholds,'minimumMargin':margin,'supportGate':gate,'silenceRms':.0005,'modelFile':'respira.onnx','limitations':report['limitations']}
 (out/'config.json').write_text(json.dumps(config));(out/'metrics.json').write_text(json.dumps(report,indent=2));np.savez_compressed(out/'parity_fixture.npz',wave=crop(waves[0]),features=first[0]);(out/'test_predictions.json').write_text(json.dumps([{'file':Path(rows[i]['path']).name,'truth':LABELS[y[i]],'scores':p.tolist()} for i,p in zip(ti,tp)],indent=2));print('FINAL',json.dumps({'recording_accuracy':report['test']['accuracy'],'first_accuracy':report['first_window_evaluation']['test']['accuracy'],'selective':report['first_window_evaluation']['selective'],'fit':report['fit_diagnostics']}),flush=True)
if __name__=='__main__':main()


Overwriting /content/respira/ml/train_respiratory.py


### 02.3 · We define the evaluation

We audit our exported ONNX model on the **first complete four-second window** of each eligible test recording. We apply the app's silence, clipping, similarity, score-margin, and class-threshold checks before counting accepted labels.

| Our measure | Our calculation |
|:---|:---|
| Overall accuracy | We divide correct top-category predictions by all eligible recordings. |
| Accepted-label precision | We divide correct accepted predictions by all accepted predictions. |
| Coverage | We divide accepted predictions by all eligible recordings. |
| Recall after abstention | We divide correct accepted predictions for a category by its actual recordings. |

We save per-recording predictions, error counts, a majority-class baseline, calibration diagnostics, and descriptive uncertainty intervals. We resample filename groups for those intervals; we do not treat them as verified patient identities.

We save this audit in `ml/audit_evaluation.py`. We reproduce the browser's decision rules in Python/ONNX here; we do not claim an end-to-end browser or device validation from this audit.


In [ ]:
%%writefile /content/respira/ml/audit_evaluation.py
"""We audit our frozen model with the app's first-window decision rules.
We keep model weights and class thresholds fixed and use no external audio.
We bootstrap filename groups without assuming verified patient identities.
We run: python ml/audit_evaluation.py MODEL_OUTPUT
"""
import csv,hashlib,json,sys
from pathlib import Path
import numpy as np
import onnxruntime as ort
from audio_features import read_audio,crop,features,softmax,N

def audit(out):
 out=Path(out);cfg=json.loads((out/'config.json').read_text());report=json.loads((out/'metrics.json').read_text());rows=json.loads((out/'manifest.json').read_text());labels=cfg['labels'];k=len(labels)
 policy_path=out/'audio-policy.json'
 policy=json.loads(policy_path.read_text()) if policy_path.exists() else {'version': 'mic-preview-v1', 'noSignalRms': 1e-07, 'noSignalPeak': 1e-06, 'quietWarningRms': 0.0005, 'clippedFraction': 0.08, 'unfamiliarAudio': 'preview_only', 'uncertainAudio': 'preview_only', 'captureAutoGainControl': True, 'note': 'Engineering input policy, not a clinically calibrated noise detector. Quiet nonzero signals are normalized by existing feature preprocessing. Preview scores do not bypass the original category acceptance thresholds.'}
 policy_path.write_text(json.dumps(policy,indent=2))
 options=ort.SessionOptions();options.intra_op_num_threads=1;session=ort.InferenceSession(str(out/'respira.onnx'),sess_options=options,providers=['CPUExecutionProvider'])
 results=[];probabilities=[];excluded=[]
 # We apply our model and acceptance rules to each complete first window.
 for r in rows:
  if r['split']!='test':continue
  wave=read_audio(r['path'])
  if len(wave)<N:excluded.append({'recording':Path(r['path']).name,'reason':'shorter than four seconds','label':labels[r['label']]});continue
  x=crop(wave);f=features(x);logits=session.run(None,{'features':f[None]})[0];p=softmax(logits/cfg['temperature'])[0];pred=int(p.argmax());ordered=np.sort(p)
  means=f[0].mean(1);sd=f[0].std(1);d=np.concatenate([means,sd]);gate=cfg['supportGate'];distance=float(np.sqrt(np.min(np.mean(((d-np.array(gate['centroids']))/np.array(gate['scale']))**2,axis=1))))
  ac=x.astype(np.float64)-x.astype(np.float64).mean();rms=float(np.sqrt(np.mean(ac*ac)));peak=float(np.abs(ac).max());clipped=float((np.abs(x)>.99).mean())
  reason='no_signal' if rms<policy['noSignalRms'] or peak<policy['noSignalPeak'] else 'clipping' if clipped>policy['clippedFraction'] else 'unsupported_audio' if distance>gate['maxDistance'] else 'below_class_threshold' if p[pred]<cfg['thresholds'][pred] else 'ambiguous_scores' if ordered[-1]-ordered[-2]<cfg['minimumMargin'] else 'accepted'
  results.append({'recording':Path(r['path']).name,'group':r['group'],'truth':labels[r['label']],'predicted':labels[pred],'shown_in_app':labels[pred] if reason=='accepted' else 'abstain','reason':reason,'exploratory_preview':reason not in ['accepted','no_signal','clipping'],'quiet_signal':rms<policy['quietWarningRms'],'top_score':float(p[pred]),'rms':rms,'clipped_fraction':clipped,'support_distance':distance});probabilities.append(p)
 y=np.array([labels.index(r['truth']) for r in results]);pred=np.array([labels.index(r['predicted']) for r in results]);take=np.array([r['reason']=='accepted' for r in results]);groups=np.array([r['group'] for r in results]);p=np.stack(probabilities)
 raw=np.bincount(y*k+pred,minlength=k*k).reshape(k,k);shown=np.bincount(y*(k+1)+np.where(take,pred,k),minlength=k*(k+1)).reshape(k,k+1)
 def ratio(a,b):return float(a/b) if b else None
 def summarize(raw,shown):
  total=int(raw.sum());accepted=int(shown[:,:k].sum());correct=int(np.trace(raw));accepted_correct=int(np.trace(shown[:,:k]));classes=[]
  for c,label in enumerate(labels):
   count=int(raw[c].sum());npred=int(raw[:,c].sum());nshow=int(shown[:,c].sum());tp=int(shown[c,c])
   classes.append({'label':label,'support':count,'raw_true_positives':int(raw[c,c]),'raw_predictions':npred,'raw_false_positives':npred-int(raw[c,c]),'raw_false_negatives':count-int(raw[c,c]),'raw_precision':ratio(raw[c,c],npred),'raw_recall':ratio(raw[c,c],count),'accepted_predictions':nshow,'accepted_false_negatives':count-tp,'wrong_accepted_for_true_class':count-tp-int(shown[c,-1]),'accepted_true_positives':tp,'accepted_false_positives':nshow-tp,'accepted_precision':ratio(tp,nshow),'recall_after_abstention':ratio(tp,count),'abstentions_for_true_class':int(shown[c,-1])})
  return {'recordings':total,'correct':correct,'accuracy':ratio(correct,total),'accepted':accepted,'accepted_correct':accepted_correct,'accepted_errors':accepted-accepted_correct,'accepted_precision':ratio(accepted_correct,accepted),'coverage':ratio(accepted,total),'abstained':total-accepted,'classes':classes}
 # We estimate descriptive uncertainty by resampling filename groups.
 summary=summarize(raw,shown);unique=np.unique(groups);raw_groups=[];shown_groups=[]
 for group in unique:
  mask=groups==group;raw_groups.append(np.bincount(y[mask]*k+pred[mask],minlength=k*k).reshape(k,k));shown_groups.append(np.bincount(y[mask]*(k+1)+np.where(take[mask],pred[mask],k),minlength=k*(k+1)).reshape(k,k+1))
 raw_groups=np.stack(raw_groups);shown_groups=np.stack(shown_groups);rng=np.random.default_rng(20261114);boots=[]
 for _ in range(2000):
  ix=rng.integers(0,len(unique),len(unique));boots.append(summarize(raw_groups[ix].sum(0),shown_groups[ix].sum(0)))
 def interval(values):
  valid=[v for v in values if v is not None];return {'low':float(np.percentile(valid,2.5)) if valid else None,'high':float(np.percentile(valid,97.5)) if valid else None,'valid_resamples':len(valid)}
 intervals={metric:interval([b[metric] for b in boots]) for metric in ['accuracy','accepted_precision','coverage']}
 for i,c in enumerate(summary['classes']):c['accepted_precision_interval']=interval([b['classes'][i]['accepted_precision'] for b in boots])
 # We compare our model scores with observed accuracy in ten score ranges.
 confidence=p.max(1);correct=(pred==y);bins=[];ece=0
 for i in range(10):
  mask=(confidence>=i/10)&(confidence<(i+1)/10 if i<9 else confidence<=1);count=int(mask.sum());avg=float(confidence[mask].mean()) if count else None;acc=float(correct[mask].mean()) if count else None
  if count:ece+=count/len(y)*abs(avg-acc)
  bins.append({'lower':i/10,'upper':(i+1)/10,'count':count,'mean_score':avg,'accuracy':acc})
 baseline=int(np.bincount([r['label'] for r in rows if r['split']=='train'],minlength=k).argmax())
 comparison={'old_report_first_window_accuracy':report['first_window_evaluation']['test']['accuracy'],'old_report_accepted':report['first_window_evaluation']['selective']['accepted'],'current_app_accepted':summary['accepted'],'note':'Current audit includes silence and clipping checks as well as support distance, margin and class thresholds. Original model selection report did not include the two signal-quality checks.'}
 assert abs(summary['accuracy']-comparison['old_report_first_window_accuracy'])<1e-10
 result={'schema_version':1,'model_version':cfg['version'],'audit_date':'2026-11-14','status':'Internal research benchmark; not clinical validation','protocol':{'implementation':'Python/ONNX audit of the browser decision rules; soundfile/SciPy decoding and resampling. This is not an end-to-end trial across browsers or microphones.','unit':'One first complete four-second window per eligible test recording','recordings':len(results),'excluded':excluded,'split_counts':{s:sum(r['split']==s for r in rows) for s in ['train','validation','test']},'group_count':len(unique),'groups':unique.tolist(),'patient_identity_verified':False,'test_previously_inspected':True,'external_audio_used':False,'model_or_class_thresholds_changed':False,'input_policy_version':policy['version'],'display_note':'Accepted-label metrics exclude exploratory score previews. Near-digital silence blocks analysis; quiet signals are analyzed. Unfamiliar and uncertain audio may show unvalidated previews without assigning a category.','confidence_interval_method':'Descriptive 95% percentile cluster bootstrap, 2,000 resamples, seed 20261114. Resamples P-number filename groups with replacement. Groups are unverified source/patient proxies; intervals do not establish clinical confidence or independent external validation. Per-class precision intervals omit resamples with zero accepted predictions for that class; valid counts are recorded.'},'summary':summary,'intervals':intervals,'confusion':{'labels':labels,'all_predictions':raw.tolist(),'app_display_columns':labels+['abstain'],'app_display':shown.tolist()},'baseline':{'policy':'Always predict the largest training class','label':labels[baseline],'correct':int((y==baseline).sum()),'accuracy':float((y==baseline).mean())},'calibration':{'ece_10_equal_width_bins':float(ece),'multiclass_brier_score':float(np.mean(np.sum((p-np.eye(k)[y])**2,axis=1))),'negative_log_likelihood':float(-np.mean(np.log(p[np.arange(len(y)),y]+1e-9))),'bins':bins,'note':'Calibration describes dataset-label predictions before abstention. These are not probabilities of disease in a person.'},'abstention_reasons':{reason:sum(r['reason']==reason for r in results) for reason in sorted(set(r['reason'] for r in results)) if reason!='accepted'},'fit_diagnostics':report['fit_diagnostics'],'selected_members':report['selected_members'],'thresholds':report['selective_policy']['threshold_details'],'earlier_report_comparison':comparison,'artifacts':{'audio_policy_sha256':hashlib.sha256(policy_path.read_bytes()).hexdigest(),'model_sha256':hashlib.sha256((out/'respira.onnx').read_bytes()).hexdigest(),'config_sha256':hashlib.sha256((out/'config.json').read_bytes()).hexdigest(),'dataset_split_sha256':hashlib.sha256(json.dumps(sorted((r['sha256_pcm'],r['label'],r['split']) for r in rows),separators=(',',':')).encode()).hexdigest()},'limitations':['No external microphone or clinical validation.','Repeated use of this test benchmark can introduce development bias.','P-number groups may not correspond to unique patients.','Few groups and small per-class counts make intervals unstable.','Abstention can improve displayed precision while missing true conditions.','No labeled noise set: noise-detection performance is unknown.']}
 # We benchmark our model against always predicting the largest training class.
 baseline_raw=np.zeros((k,k),dtype=int);baseline_raw[:,baseline]=raw.sum(1)
 def macro_f1(matrix):
  denom=matrix.sum(0)+matrix.sum(1)
  return float(np.mean(np.divide(2*np.diag(matrix),denom,out=np.zeros(k,dtype=float),where=denom!=0)))
 result['baseline'].update(recordings=len(results),macro_f1=macro_f1(baseline_raw),balanced_accuracy=1/k,confusion_matrix=baseline_raw.tolist())
 result['comparison']={'accuracy_gain_percentage_points':100*(summary['accuracy']-result['baseline']['accuracy']),'additional_correct':summary['correct']-result['baseline']['correct'],'model_macro_f1':macro_f1(raw),'model_balanced_accuracy':float(np.mean(np.diag(raw)/raw.sum(1))),'note':'Both methods predict all 218 eligible first windows; selective precision is not substituted for accuracy.'}
 result['candidate_comparison']=[{'name':c['name'],'validation_accuracy':c['validation']['accuracy'],'validation_macro_f1':c['validation']['macro_f1'],'selected':c['name'] in report['selected_members']} for c in report['model_selection']['candidates']]
 for choice in report['model_selection']['choices']:
  if len(choice['members'])>1:result['candidate_comparison'].append({'name':str(len(choice['members']))+'-model ensemble','validation_accuracy':choice['validation']['accuracy'],'validation_macro_f1':choice['validation']['macro_f1'],'selected':False})
 failure_index=next((i for i,r in enumerate(results) if r['reason']=='accepted' and r['truth']!=r['predicted']),None)
 result['failure_example']=None if failure_index is None else {**results[failure_index],'window_seconds':[0,4],'scores':dict(zip(labels,map(float,p[failure_index]))),'selection_rule':'First accepted misclassification in manifest order; illustrative, not representative of every failure.','interpretation':'The acceptance checks did not prevent this wrong category. Overlapping acoustic patterns or limited representative data are possible explanations, not established causes.'}
 result['brief_alignment']={'track':'AI + Health — Our own idea','intended_user':'An adult curious about respiratory sound patterns, trying a short recording at home.','supported_use':'Research exploration of dataset-label similarities; home-microphone screening benefit is not established.','demographic_evaluation':'Not performed: verified age and sex metadata were not used. Category breakdowns are not demographic fairness tests.','disclaimer':'This project is a prototype and is not medical advice.','data_policy':'Public respiratory dataset only. No private patient records or newly collected medically labeled recordings are used.'}
 (out/'evaluation.json').write_text(json.dumps(result,indent=2))
 with (out/'evaluation-predictions.csv').open('w') as f:
  writer=csv.DictWriter(f,fieldnames=list(results[0]),lineterminator='\n');writer.writeheader();writer.writerows(results)
 print(json.dumps({'summary':summary,'group_count':len(unique),'intervals':intervals,'comparison':comparison},indent=2))
 return result
if __name__=='__main__':audit(sys.argv[1])


Overwriting /content/respira/ml/audit_evaluation.py


## 03 · We train and audit
### 03.1 · Our training run

We train our candidates, keep each candidate's best validation checkpoint, and select the strongest validation macro F1. We save the selected model, configuration, split manifest, and metrics in `training-respiratory-output`.

We evaluate the internal test recordings after fixing our selection and thresholds. We retain the earlier-development limitation described in section 01.


In [ ]:
# We run our candidate search, selection, calibration, and internal evaluation.
subprocess.run([sys.executable,'-u','ml/train_respiratory.py',
                '--data','data','--respiratory-root',str(RESPIRATORY_ROOT),
                '--out','training-respiratory-output'],check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', 'ml/train_respiratory.py', '--data', 'data', '--respiratory-root', '/content/respira/data/asthma-v2', '--out', 'training-respiratory-output'], returncode=0)

### 03.2 · Our app-policy audit

We load the exported model, apply our audio-quality and acceptance rules, and save `evaluation.json` and `evaluation-predictions.csv`. We use these files to distinguish raw predictions, accepted categories, and withheld labels in our results.


In [ ]:
# We audit our exported model with the app's category-display policy.
subprocess.run([sys.executable,'ml/audit_evaluation.py','training-respiratory-output'],check=True)


CompletedProcess(args=['/usr/bin/python3', 'ml/audit_evaluation.py', 'training-respiratory-output'], returncode=0)

## 04 · We inspect our results

We compare training progress, validation performance, per-category errors, and acceptance coverage. We mark each candidate's selected epoch on its validation plot.

We interpret augmented training loss separately from unaugmented validation loss because we also use label smoothing during training. We use the unaugmented `fit_diagnostics` values for a closer train-versus-validation comparison.

We include every eligible test recording in our raw confusion matrix, including recordings for which we later withhold a category. We report accepted-label precision alongside coverage so we can see the cost of withholding predictions.


In [ ]:
# We inspect our current run: metrics, training curves, and raw confusion matrix.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
out=Path('training-respiratory-output')
r=json.loads((out/'metrics.json').read_text())
print('Selected:',r['selected_members'])
print('All-recording accuracy:',r['test']['accuracy'])
print('First-four-second accuracy:',r['first_window_evaluation']['test']['accuracy'])
display(pd.DataFrame(r['first_window_evaluation']['test']['report']).T)
audit=json.loads((out/'evaluation.json').read_text())
selected=audit['summary']
print('Precision among accepted first-window category labels:',selected['accepted_precision'])
print('Coverage:',selected['coverage'],'Accepted:',selected['accepted'],'of',selected['recordings'])
display(pd.DataFrame(selected['classes']).set_index('label'))
display(pd.DataFrame(r['selective_policy']['threshold_details']))
print('Train/validation comparison:',r['fit_diagnostics'])
fig,axes=plt.subplots(len(r['model_selection']['candidates']),2,figsize=(12,10))
for row,candidate in enumerate(r['model_selection']['candidates']):
    h=pd.DataFrame(candidate['history'])
    axes[row,0].plot(h.epoch,h.training_augmented_loss,label='Augmented training loss')
    axes[row,0].plot(h.epoch,h.validation_loss,label='Unaugmented validation loss')
    axes[row,0].set(title=candidate['name'],xlabel='Epoch',ylabel='Loss')
    axes[row,0].legend()
    axes[row,1].plot(h.epoch,h.validation_accuracy,label='Validation accuracy')
    axes[row,1].plot(h.epoch,h.validation_macro_f1,label='Validation macro F1')
    axes[row,1].axvline(candidate['best_epoch'],linestyle='--',color='gray')
    axes[row,1].set(xlabel='Epoch',ylim=(0,1))
    axes[row,1].legend()
plt.tight_layout();plt.show()
print('Training curve uses augmentation and label smoothing, so its loss is not directly comparable to validation loss. Use the unaugmented fit_diagnostics comparison above.')
cm=np.array(r['first_window_evaluation']['test']['confusion_matrix'])
fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=range(5),yticks=range(5),xticklabels=r['labels'],yticklabels=r['labels'],xlabel='Predicted',ylabel='Actual',title='First four seconds — all predictions, including abstentions')
for i in range(5):
    for j in range(5):ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()


## 05 · We export our model

We package our ONNX weights, configuration, metrics, audit, per-recording predictions, and audio policy in `respira-dataset-only-model.zip`.

We deploy the weights and matching settings together to preserve our preprocessing and acceptance rules. We update the live app through a separate deployment step; we do not change it by rerunning this notebook.


In [ ]:
# We export our model and matching evaluation artifacts as one package.
with zipfile.ZipFile('respira-dataset-only-model.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['respira.onnx','config.json','metrics.json','evaluation.json','evaluation-predictions.csv','audio-policy.json']:
        z.write(out/name,arcname=name)
print('Export:',ROOT/'respira-dataset-only-model.zip')
# We can download our ZIP with google.colab.files.download after export.


Export: /content/respira/respira-dataset-only-model.zip


## 06 · Our saved results

> **Our reference run · November 14, 2026**  
> We report the saved local run below. We keep these reference figures separate from any new Colab execution.

| Our measure | Our result | Our evaluation scope |
|:---|---:|:---|
| First-window accuracy | **78.0%** | 170 correct / 218 eligible recordings |
| Accepted-label precision | **88.1%** | 111 correct / 126 accepted predictions |
| Coverage | **57.8%** | 126 accepted / 218 eligible recordings |
| Combined-window accuracy | **79.5%** | First/last-window combination across 219 test recordings |

We selected **`wide-crops`**. We reserved 219 test recordings and evaluated 218 in the first-window audit after excluding one recording shorter than four seconds. We included silence and clipping checks in the accepted-label results.

We made **48 raw classification errors** across the 218 eligible recordings. After our acceptance checks, we retained **111 correct and 15 incorrect labels** and withheld a category on **92 recordings**.

We did not reach our 90% overall accuracy target. We report 88.1% only as precision among accepted labels, together with its 57.8% coverage.


### 06.1 · We compare model candidates

| Our candidate | Validation accuracy | Validation macro F1 |
|:---|---:|---:|
| `wide` | 84.5% | 0.809 |
| **`wide-crops` · selected** | **86.4%** | **0.818** |
| `compact` | 82.6% | 0.801 |
| Two-model ensemble | 86.4% | 0.817 |
| Three-model ensemble | 85.3% | 0.804 |

We selected `wide-crops` using **265 validation recordings**. We report validation performance here and test performance in our results summary above.

We changed cropping, dropout, learning rate, and random seed together between the wide configurations. We therefore compare configurations as a whole; we cannot attribute the gain to cropping alone.


### 06.2 · We compare against a baseline

| Our method | Correct / total | Accuracy | Macro F1 |
|:---|---:|---:|---:|
| Majority-class baseline: always COPD | 72 / 218 | 33.0% | 0.099 |
| Our selected CNN | 170 / 218 | 78.0% | 0.723 |

We gained **45.0 percentage points** in accuracy, or **98 additional correct classifications**, over always predicting the largest training class. We evaluated both methods on the same first four seconds of each eligible test recording.

We use this comparison to measure improvement over a simple baseline. We keep our 88.1% accepted-label precision separate because we obtain it on only 57.8% of recordings.


### 06.3 · We examine errors by category

**Before we apply acceptance checks**

| Category | Actual clips | False positives | False negatives | Recall |
|:---|---:|---:|---:|---:|
| Asthma | 52 | 11 | 10 | 80.8% |
| Healthy | 20 | 10 | 2 | 90.0% |
| Bronchial | 20 | 8 | 13 | 35.0% |
| COPD | 72 | 10 | 15 | 79.2% |
| Pneumonia | 54 | 9 | 8 | 85.2% |

We evaluate each category against all others. We count a wrong label as a false positive for the predicted category and a false negative for the actual category.

**After we apply acceptance checks**

| Category | False positives | False negatives* | Wrong accepted label | Withheld label | Recall |
|:---|---:|---:|---:|---:|---:|
| Asthma | 1 | 33 | 2 | 31 | 36.5% |
| Healthy | 0 | 20 | 2 | 18 | 0.0% |
| Bronchial | 6 | 14 | 3 | 11 | 30.0% |
| COPD | 7 | 20 | 7 | 13 | 72.2% |
| Pneumonia | 1 | 20 | 1 | 19 | 63.0% |

\* We include wrong accepted labels and withheld labels in false negatives for the actual category. We group the last three columns by the recording's true category.

We reduce false positives overall, but we also miss more true categories. We return no correct accepted Healthy labels in this run, and we retain only **30.0% Bronchial recall** after acceptance checks. We identify these gaps as priorities for further evaluation.

We use abstention to withhold a category, not to indicate a negative diagnosis. We exclude exploratory score previews from accepted-label metrics.


### 06.4 · We inspect an accepted mistake

| Our example | Saved result |
|:---|:---|
| Recording | `P2BronchialTc_9.wav` |
| Audio window | First four seconds |
| Dataset label | Bronchial |
| Our accepted prediction | COPD |
| Our top model score | 69.9% |

We accepted the wrong category despite our checks. We interpret 69.9% as a model score, not a person's disease probability.

We consider overlapping acoustic patterns and limited representative data as possible explanations, but we have not established the cause. We use this example to show why acceptance checks do not eliminate mistakes.

We select the first accepted misclassification in manifest order. We use it as one inspectable example rather than a summary of all failures, and we provide playback in section 06.6.


### 06.5 · What we can conclude

We improved substantially over our majority-class baseline on this internal dataset. We also increased precision among displayed categories by withholding uncertain or unsupported predictions, while reducing how often we returned a correct category.

We use our scores to describe similarity to dataset labels. We have not established a person's disease probability, clinical diagnostic performance, or demographic fairness. We treat **Healthy** as a dataset category, not a confirmation of health.

We still need independent recordings, verified patient splits, and broader device testing before we can assess performance beyond this benchmark.


### 06.6 · We verify the latest run

We load the current audit, compare our CNN with the majority-class baseline, and check false-positive and false-negative counts against `evaluation-predictions.csv`.

We then display and play the first accepted mistake, if we have one in the current run. We use this cell to connect our summary metrics to a recording we can inspect directly.


In [ ]:
# We verify our current metrics against the saved per-recording predictions.
import pandas as pd
current=json.loads((out/'evaluation.json').read_text())
s=current['summary'];b=current['baseline'];labels=current['confusion']['labels']
display(pd.DataFrame([
    {'Method':'Always '+b['label'],'Correct':b['correct'],'Total':s['recordings'],'Accuracy':b['accuracy'],'Macro F1':b['macro_f1']},
    {'Method':'Selected CNN','Correct':s['correct'],'Total':s['recordings'],'Accuracy':s['accuracy'],'Macro F1':current['comparison']['model_macro_f1']}
]))
display(pd.DataFrame(s['classes'])[['label','support','raw_false_positives','raw_false_negatives','raw_recall','accepted_false_positives','accepted_false_negatives','wrong_accepted_for_true_class','abstentions_for_true_class','recall_after_abstention']])
display(pd.DataFrame(current['candidate_comparison']))
# We check error counts against the saved predictions.
predictions=pd.read_csv(out/'evaluation-predictions.csv')
for c in s['classes']:
    actual=predictions.truth.eq(c['label']);predicted=predictions.predicted.eq(c['label'])
    accepted=predictions.shown_in_app.eq(c['label'])
    assert int((~actual & predicted).sum()) == c['raw_false_positives']
    assert int((actual & ~predicted).sum()) == c['raw_false_negatives']
    assert int((~actual & accepted).sum()) == c['accepted_false_positives']
    assert int((actual & ~accepted).sum()) == c['accepted_false_negatives']
example=current['failure_example']
if example:
    display(pd.DataFrame([example]).drop(columns=['scores']))
    display(pd.Series(example['scores'],name='Model score'))
    # We play the first four seconds of this mistaken prediction.
    from IPython.display import Audio, display
    from ml.audio_features import read_audio, crop, SR
    manifest=json.loads((out/'manifest.json').read_text())
    source=next(r for r in manifest if r['split']=='test' and Path(r['path']).name==example['recording'])
    display(Audio(crop(read_audio(source['path'])),rate=SR))
else:
    print('No accepted misclassification in this run.')


,Method,Correct,Total,Accuracy,Macro F1
0,Always copd,72,218,0.330275,0.09931
1,Selected CNN,173,218,0.793578,0.73835


,label,support,raw_false_positives,raw_false_negatives,raw_recall,accepted_false_positives,accepted_false_negatives,wrong_accepted_for_true_class,abstentions_for_true_class,recall_after_abstention
0,asthma,52,10,10,0.807692,6,23,5,18,0.557692
1,healthy,20,12,1,0.950000,0,20,1,19,0.000000
2,bronchial,20,9,12,0.400000,8,15,6,9,0.250000
3,copd,72,7,14,0.805556,6,21,10,11,0.708333
4,pneumonia,54,7,8,0.851852,3,20,1,19,0.629630


,name,validation_accuracy,validation_macro_f1,selected
0,wide,0.833962,0.798567,True
1,wide-crops,0.852830,0.809083,True
2,compact,0.830189,0.802838,False
3,2-model ensemble,0.860377,0.811506,False
4,3-model ensemble,0.849057,0.800227,False


,recording,group,truth,predicted,shown_in_app,reason,exploratory_preview,quiet_signal,top_score,rms,clipped_fraction,support_distance,window_seconds,selection_rule,interpretation
0,P12BronchialSc_56.wav,resp-P12,bronchial,asthma,asthma,accepted,False,False,0.862078,0.094009,0.0,0.884874,"[0, 4]",First accepted misclassification in manifest o...,The acceptance checks did not prevent this wro...


,Model score
asthma,0.862078
healthy,0.056292
bronchial,0.020860
copd,0.002764
pneumonia,0.058006


## 07 · Our contribution and next steps

### What we built

We connect a reproducible respiratory-audio pipeline with browser inference and a usable upload flow. We combine our accepted model findings with optional symptom context in the app's explanation layer. We keep classification, acceptance decisions, and contextual explanation distinct.

We make our evaluation inspectable through a same-data baseline, per-category errors, coverage, and a playable failure example. We use public training data and keep our separate microphone and archived TensorFlow experiments outside the current CNN results.

### What we will improve

| Our priority | Our next step |
|:---|:---|
| Independent validation | We will evaluate on an external stethoscope dataset with verified patient splits and reserve its test set from tuning. |
| Weak category recall | We will investigate Bronchial confusion and the absence of correct accepted Healthy labels. |
| Device robustness | We will test across recording devices and acquisition conditions. |
| Noise handling | We will collect suitable noise labels before measuring or claiming noise-detection performance. |
| Fairness and clinical relevance | We will evaluate demographic subgroups when appropriate verified metadata are available. |

### Our sources and development support

- **Our data:** We use Mohammed Tawfik Musaed's [Asthma Detection Dataset](https://www.kaggle.com/datasets/mohammedtawfikmusaed/asthma-detection-dataset-version-2).
- **Our tools:** We use [PyTorch](https://pytorch.org/), [ONNX Runtime](https://onnxruntime.ai/), [scikit-learn](https://scikit-learn.org/), [NumPy](https://numpy.org/), [SciPy](https://scipy.org/), [SoundFile](https://python-soundfile.readthedocs.io/), [pandas](https://pandas.pydata.org/), and [Matplotlib](https://matplotlib.org/).
- **Our development support:** We used [OpenAI Codex](https://openai.com/codex/) for debugging, code refinement, and app development.
- **Our submission reference:** We reference the Veritas AI International High School Hackathon 2026 Project Briefs, sections 3, 4.1, 5, and 6.

---


## Appendix · Our earlier experiments

We preserve our original TensorFlow/Conv1D–LSTM exploration below as development history. We exclude these experiments from our current CNN pipeline, selection, and reported benchmark.

We keep `RUN_LEGACY_NOTEBOOK = False` during the main workflow. We treat the archive as exploratory code with its own dependencies and dataset path.

### A.1 · We explore recordings and features

We inspect category counts, waveforms, spectrograms, recording lengths, and summary features.


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We import the analysis libraries.
    import os
    import librosa
    import librosa.display
    import numpy as np
    import matplotlib.pyplot as plt
    from prettytable import PrettyTable
    import seaborn as sns
    import pandas as pd


    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder, label_binarize
    from sklearn.decomposition import PCA

    from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import (
        Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Dropout,
        Bidirectional, Attention, Input, LayerNormalization
    )
    import random
    # We require SHAP in the environment only when running this archive.

    import shap


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We set the audio folder.
    dataset_path = '/content/Asthma_Detection_Dataset'

    # We list the five label folders.
    categories = ['Bronchial', 'pneumonia', 'asthma', 'healthy', 'copd']

    print("Categories:", categories)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We count WAV files in each category and plot the totals.
    category_counts = {
        category: len([
            f for f in os.listdir(os.path.join(dataset_path, category))
            if f.endswith('.wav')
        ])
        for category in categories
    }

    plt.figure(figsize=(8, 6))
    plt.bar(category_counts.keys(), category_counts.values())
    plt.xlabel('Category')
    plt.ylabel('Number of Files')
    plt.title('Number of Audio Files in Each Category')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We show recording counts in a table.
    table = PrettyTable()
    table.field_names = ["Category", "Number of Audio Files"]
    for category, count in category_counts.items():
        table.add_row([category, count])

    print("Number of Audio Files in Each Category:")
    print(table)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot one waveform per category.
    print("\n=== Waveform Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                plt.figure(figsize=(10, 4))
                librosa.display.waveshow(y, sr=sr)
                plt.title(f'Waveform - {category}')
                plt.xlabel('Time (s)')
                plt.ylabel('Amplitude')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot one mel spectrogram per category.
    print("\n=== Mel Spectrogram Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                mel_spect = librosa.feature.melspectrogram(y=y, sr=sr)
                mel_spect_db = librosa.power_to_db(mel_spect, ref=np.max)

                plt.figure(figsize=(10, 4))
                librosa.display.specshow(mel_spect_db, sr=sr, x_axis='time', y_axis='mel')
                plt.colorbar(format='%+2.0f dB')
                plt.title(f'Mel Spectrogram - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot 13 MFCC features for one clip per category.
    print("\n=== MFCC Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
                plt.figure(figsize=(10, 4))
                librosa.display.specshow(mfccs, sr=sr, x_axis='time')
                plt.colorbar()
                plt.title(f'MFCCs - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot spectrograms on a log-frequency axis.
    print("\n=== Log Power Spectrogram Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                D = np.abs(librosa.stft(y))**2
                log_power_spectrogram = librosa.power_to_db(D, ref=np.max)

                plt.figure(figsize=(10, 4))
                librosa.display.specshow(
                    log_power_spectrogram, sr=sr, x_axis='time', y_axis='log'
                )
                plt.colorbar(format='%+2.0f dB')
                plt.title(f'Log Power Spectrogram - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot how often each waveform crosses zero.
    print("\n=== Zero Crossing Rate Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                zero_crossings = librosa.feature.zero_crossing_rate(y)[0]

                plt.figure(figsize=(10, 4))
                plt.plot(zero_crossings, label='Zero Crossing Rate')
                plt.xlabel('Frames')
                plt.ylabel('Rate')
                plt.title(f'Zero-Crossing Rate - {category}')
                plt.legend()
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot pitch-class features for one clip per category.
    print("\n=== Chroma Feature Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                chroma = librosa.feature.chroma_stft(y=y, sr=sr)
                plt.figure(figsize=(10, 4))
                librosa.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma')
                plt.colorbar()
                plt.title(f'Chroma Features - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We compare recording lengths by category.
    durations = []
    labels = []

    print("\n=== Distribution of Audio Durations by Category ===")
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                audio_path = os.path.join(category_path, audio_file)
                y, sr = librosa.load(audio_path, sr=None)
                durations.append(librosa.get_duration(y=y, sr=sr))
                labels.append(category)

    df = pd.DataFrame({'Duration': durations, 'Category': labels})
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='Category', y='Duration', data=df)
    plt.title('Distribution of Audio Durations by Category')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot MFCC features for the last audio clip loaded in this archive.
    mfcc_sample = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    plt.figure(figsize=(10, 6))
    sns.heatmap(mfcc_sample, cmap='coolwarm')
    plt.title('MFCC Heatmap')
    plt.show()



In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We compare average MFCC features by category.
    avg_mfccs = []

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        mfccs_for_cat = []
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_cat, sr_cat = librosa.load(os.path.join(category_path, audio_file), sr=None)
                mfcc_tmp = librosa.feature.mfcc(y=y_cat, sr=sr_cat, n_mfcc=13)
                mfccs_for_cat.append(np.mean(mfcc_tmp, axis=1))
        avg_mfccs.append(np.mean(mfccs_for_cat, axis=0))

    plt.figure(figsize=(10, 6))
    for i, avg_mfcc in enumerate(avg_mfccs):
        plt.plot(avg_mfcc, label=categories[i])
    plt.title('Average MFCCs Across Categories')
    plt.xlabel('MFCC Coefficients')
    plt.ylabel('Average Value')
    plt.legend()
    plt.show()



### A.2 · We compare feature summaries

We continue our archived exploration with waveform overlays, a two-component PCA projection, spectral centroids, and aggregate spectral magnitude. We use these plots for exploration rather than as evidence for our current CNN's performance.


In [ ]:
if RUN_LEGACY_NOTEBOOK:

    # We overlay one waveform per category.
    print("\n=== Amplitude Over Time (One sample per category) ===")
    plt.figure(figsize=(10, 5))
    for category in categories:
        audio_path = os.path.join(dataset_path, category, os.listdir(os.path.join(dataset_path, category))[0])
        y_cat, sr_cat = librosa.load(audio_path, sr=None)
        plt.plot(y_cat, label=category)

    plt.title('Amplitude Over Time')
    plt.xlabel('Time')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We project mean MFCC features into two PCA dimensions and color by category.
    print("\n=== PCA Visualization of Audio Features ===")
    pca_features = []
    pca_labels = []

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_pca, sr_pca = librosa.load(os.path.join(category_path, audio_file), sr=None)
                mfcc_pca = librosa.feature.mfcc(y=y_pca, sr=sr_pca, n_mfcc=13)
                pca_features.append(np.mean(mfcc_pca, axis=1))
                pca_labels.append(category)

    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(pca_features)

    df_pca = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])
    df_pca['Category'] = pca_labels

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Category', s=100)
    plt.title('PCA Visualization of Audio Features')
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:

    # We pool mean spectral centroids across categories and plot their distribution.
    print("\n=== Distribution of Spectral Centroids ===")
    centroids = []
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_cent, sr_cent = librosa.load(os.path.join(category_path, audio_file), sr=None)
                centroid = librosa.feature.spectral_centroid(y=y_cent, sr=sr_cent)[0]
                centroids.append(np.mean(centroid))

    sns.histplot(centroids, bins=20, kde=True)
    plt.title('Distribution of Spectral Centroids')
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We compare aggregate spectral magnitude by category in this archive.
    print("\n=== Average Frequency Band Energy by Category ===")
    band_energies = []
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        energies = []
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_be, sr_be = librosa.load(os.path.join(category_path, audio_file), sr=None)
                stft = np.abs(librosa.stft(y_be))
                band_energy = np.sum(stft, axis=1)
                energies.append(np.mean(band_energy))
        band_energies.append(np.mean(energies))

    plt.bar(categories, band_energies)
    plt.title('Average Frequency Band Energy by Category')
    plt.ylabel('Energy')
    plt.show()



### A.3 · We preserve our augmentation trials

We retain our early noise, time-stretch, and pitch-shift trials for reference. We build an augmented list here, but our archived feature-extraction cell below reloads the original files. We therefore do not claim that these augmentations trained the archived classifier or our current CNN.


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We define audio augmentation functions.
    def my_add_noise(audio, noise_factor=0.005):
        """We add random noise to the audio signal in our archived experiment."""
        noise = np.random.randn(len(audio))
        return audio + noise_factor * noise

    def my_time_stretch(audio, rate=1.0):
        """We time-stretch audio with librosa at our chosen rate."""
        return librosa.effects.time_stretch(y=audio, rate=rate)

    def my_pitch_shift(audio, sr, n_steps=0):
        """We shift pitch by n_steps semitones with librosa."""
        return librosa.effects.pitch_shift(y=audio, sr=sr, n_steps=n_steps)

    def augment_audio(y, sr):
        """
        We randomly choose noise, stretching, pitch shifting, or no change
        for one archived augmentation trial.
        """
        import random
        choice = random.choice(['noise', 'stretch', 'pitch', 'none'])

        if choice == 'noise':
            return my_add_noise(y)
        elif choice == 'stretch':
            rate = random.uniform(0.8, 1.2)   # We sample our stretch rate.
            return my_time_stretch(y, rate=rate)
        elif choice == 'pitch':
            steps = random.uniform(-2, 2)     # We sample our pitch shift in semitones.
            return my_pitch_shift(y, sr, n_steps=steps)
        else:
            return y  # We retain our original audio.


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We balance categories with augmented audio.
    print("\n=== Data Augmentation and Class Balancing ===")
    max_count = max(category_counts.values())

    all_audio_data = []   # We store (audio, sample rate, label) tuples.
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        current_count = len(audio_files)

        # We load the original clips.
        loaded_signals = []
        for audio_file in audio_files:
            audio_path = os.path.join(category_path, audio_file)
            y_load, sr_load = librosa.load(audio_path, sr=None)
            loaded_signals.append((y_load, sr_load))
            all_audio_data.append((y_load, sr_load, category))

        # We add augmented clips until this category matches the largest.
        if current_count < max_count:
            needed = max_count - current_count
            for _ in range(needed):
                orig_y, orig_sr = random.choice(loaded_signals)
                y_aug = augment_audio(orig_y, orig_sr)
                all_audio_data.append((y_aug, orig_sr, category))



### A.4 · We preserve our earlier classifier

We extract mean MFCC features from the original recordings, use an 80/20 random split, and fit our earlier Conv1D–LSTM. We keep its plots and metrics outside our current benchmark because we do not apply our current grouping and evaluation protocol in this archive.


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We extract mean MFCCs from original files; we do not read all_audio_data here.
    print("\n=== Preparing Features and Labels for Model ===")
    feature_list = []
    labels_list = []

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        for audio_file in audio_files:
            audio_path = os.path.join(category_path, audio_file)
            y_data, sr_data = librosa.load(audio_path, sr=None)
            mfcc_data = librosa.feature.mfcc(y=y_data, sr=sr_data, n_mfcc=13)
            # We average each MFCC over time.
            feature_list.append(np.mean(mfcc_data, axis=1))
            labels_list.append(category)

    X = np.array(feature_list)
    y = np.array(labels_list)

    print("Total Samples:", len(feature_list))
    print("Feature Shape:", X.shape)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We convert category names to numeric labels.
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)
    print("Labels Shape:", y_encoded.shape)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We use an archived 80/20 random split without our current filename-group rule.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42
    )

    print("Training Samples:", len(X_train))
    print("Testing Samples:", len(X_test))


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We reshape features to (samples, timesteps, 1).
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We build and compile the Conv1D–LSTM model.
    model = Sequential()

    model.add(
        Conv1D(
            filters=32, kernel_size=3, activation='relu',
            input_shape=(X_train.shape[1], 1)
        )
    )
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64, return_sequences=True))
    model.add(LSTM(64))
    model.add(Dropout(0.3))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We train for 50 epochs, using 20% of training data for validation.
    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2
    )


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot training and validation accuracy.
    plt.figure(figsize=(8, 6))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Train vs. Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We measure test loss and accuracy.
    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot the confusion matrix and print class metrics.
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    plt.rcParams['figure.facecolor'] = 'w'
    plt.rcParams['axes.facecolor'] = 'w'

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title("Confusion Matrix (Test Data)", color='black')
    plt.xlabel("Predicted Label", color='black')
    plt.ylabel("True Label", color='black')
    plt.show()

    # We print precision, recall and F1 by class.
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    # We plot one ROC curve per class.
    # We compare each class against all other classes.
    n_classes = len(np.unique(y_test))
    y_test_binarized = label_binarize(y_test, classes=list(range(n_classes)))

    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(6,5))
    plt.rcParams['figure.facecolor'] = 'w'
    plt.rcParams['axes.facecolor'] = 'w'

    colors = ['red','green','blue','orange','purple']
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f'Class {encoder.classes_[i]} (AUC = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', color='black')
    plt.ylabel('True Positive Rate', color='black')
    plt.title('Multi-Class AUC-ROC', color='black')
    plt.legend(loc="lower right")
    plt.show()
